# Activity 2 — File Processing
**Module:** Advanced Programming — Week 2  
**University of York, MSc Computer Science**

**Files used:**
- `PeopleTrainingDate.csv` — main dataset
- `PeopleTrainingDateUpdate.csv` — update file with different column order

---
## Exercise 1: Parse CSV using string functions only and print formatted table

**Constraint:** string functions only — no `csv` module, no `pandas`.  
Parsing strategy: `file.read()` → `strip().split('\n')` for rows → `split(',')` for columns.

**Design note:** records are stored as a list of dictionaries. This makes downstream sorting and field access explicit by name (`r['Updated']`) rather than by index (`r[5]`), which would be brittle if column order ever changes.

In [ ]:
# --- Exercise 1: Parse and display ---

STANDARD_FIELDS = ['Title', 'Name', 'ID', 'Email', 'Company', 'Updated']

def parse_csv_string(filepath):
    """
    Parse a CSV file using string functions only.
    Returns a list of dicts mapped to STANDARD_FIELDS using the file's own header row.
    Raises ValueError if any required field is missing from the header.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = f.read()

    lines = data.strip().split('\n')
    headers = [h.strip() for h in lines[0].split(',')]

    # Validate all expected fields are present
    missing = [f for f in STANDARD_FIELDS if f not in headers]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Map standard field names to column indices in this file
    field_index = {field: headers.index(field) for field in STANDARD_FIELDS}

    records = []
    for row in lines[1:]:
        row = row.strip()
        if not row:          # skip blank lines
            continue
        values = row.split(',')
        if len(values) != len(headers):
            print(f"  Warning: skipping malformed row (expected {len(headers)} columns, "
                  f"got {len(values)}): {row}")
            continue
        record = {field: values[field_index[field]].strip() for field in STANDARD_FIELDS}
        records.append(record)

    return records


def print_table(records, title=''):
    """Print records as a formatted fixed-width table."""
    # Column widths: field name or widest value, whichever is larger
    widths = {f: max(len(f), max((len(r[f]) for r in records), default=0)) 
              for f in STANDARD_FIELDS}

    sep = '  '
    header_line = sep.join(f.ljust(widths[f]) for f in STANDARD_FIELDS)
    divider = sep.join('-' * widths[f] for f in STANDARD_FIELDS)

    if title:
        print(f"\n{title}")
        print('=' * len(divider))
    print(header_line)
    print(divider)
    for r in records:
        print(sep.join(r[f].ljust(widths[f]) for f in STANDARD_FIELDS))
    print(f"\n{len(records)} record(s)")


# Load and display
records = parse_csv_string('PeopleTrainingDate.csv')
print_table(records, title='PeopleTrainingDate.csv — original order')

---
## Exercise 2: Sort by date (oldest first) and write to file with 'Updated' as first column

In [ ]:
# --- Exercise 2: Sort and write to file ---

from datetime import datetime

OUTPUT_FILE = 'sorted_output.csv'
OUTPUT_FIELDS = ['Updated', 'Title', 'Name', 'ID', 'Email', 'Company']


def sort_by_date(records):
    """Return records sorted oldest-first by the 'Updated' field (YYYY-MM-DD)."""
    return sorted(records, key=lambda r: datetime.strptime(r['Updated'], '%Y-%m-%d'))


def write_csv(filepath, records, fieldnames):
    """Write records to a CSV file using string operations only."""
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(','.join(fieldnames) + '\n')
        for r in records:
            f.write(','.join(r[field] for field in fieldnames) + '\n')


sorted_records = sort_by_date(records)
write_csv(OUTPUT_FILE, sorted_records, OUTPUT_FIELDS)

print(f"Written to '{OUTPUT_FILE}'")
print_table(sorted_records, title='Sorted oldest → newest (Updated first column)')

In [ ]:
# Verify the output file was written correctly
print("Contents of", OUTPUT_FILE)
print()
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    print(f.read())

---
## Exercise 3: Process the update file and append to sorted output

**Problem:** `PeopleTrainingDateUpdate.csv` has a different column order (`ID,Email,Updated,Title,Company,Name`).  

**Solution:** `parse_csv_string()` already handles this — it reads the header row of the update file and builds a `field_index` map dynamically. No hardcoded column positions anywhere.

**Additional checks:**
- Malformed rows (wrong number of columns) → logged and skipped
- Duplicate IDs (a record in the update file that already exists in sorted_output.csv) → reported, update record overwrites the existing one

In [ ]:
# --- Exercise 3: Load update file, validate, deduplicate, re-sort, overwrite output ---

def load_sorted_output(filepath):
    """Read the existing sorted_output.csv back into memory."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = f.read()
    lines = data.strip().split('\n')
    # sorted_output.csv uses OUTPUT_FIELDS order
    existing = []
    for row in lines[1:]:
        row = row.strip()
        if not row:
            continue
        values = row.split(',')
        record = {f: values[i].strip() for i, f in enumerate(OUTPUT_FIELDS)}
        existing.append(record)
    return existing


# Step 1: Load update file (parse_csv_string handles different column order automatically)
print("Loading update file...")
update_records = parse_csv_string('PeopleTrainingDateUpdate.csv')
print(f"  Update records loaded: {len(update_records)}")
print()

# Step 2: Load existing output
existing_records = load_sorted_output(OUTPUT_FILE)
print(f"Existing records in {OUTPUT_FILE}: {len(existing_records)}")
print()

# Step 3: Build ID-keyed dict from existing records (last write wins for duplicates)
merged = {r['ID']: r for r in existing_records}

# Step 4: Apply updates — report duplicates, then overwrite
print("Applying updates:")
for r in update_records:
    if r['ID'] in merged:
        old_date = merged[r['ID']]['Updated']
        print(f"  Duplicate ID {r['ID']} ({r['Name']}): "
              f"existing Updated={old_date} → new Updated={r['Updated']} [overwritten]")
    else:
        print(f"  New record: {r['ID']} — {r['Name']}")
    merged[r['ID']] = r

# Step 5: Re-sort and write back
final_records = sort_by_date(list(merged.values()))
write_csv(OUTPUT_FILE, final_records, OUTPUT_FIELDS)

print(f"\nFinal record count: {len(final_records)}")
print_table(final_records, title=f'Final {OUTPUT_FILE} after update')

In [ ]:
# Final file contents
print(f"Contents of {OUTPUT_FILE}:")
print()
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    print(f.read())

---
## Code review notes

### Program flow and organisation
The three exercises are separated into named functions (`parse_csv_string`, `sort_by_date`, `write_csv`, `print_table`, `load_sorted_output`) rather than one monolithic script. Each function has a single responsibility and a docstring. This makes the code testable in isolation and easy to modify — for example, changing the output column order only requires updating `OUTPUT_FIELDS`.

### Effectiveness of data structures
**List of dicts** is the right structure here. Each `record` is a dict so fields are accessed by name, not by index — `r['Updated']` not `r[5]`. This means the code is robust to column reordering (Exercise 3 explicitly tests this). A list of lists would have been simpler to write but would break the moment column order changed.

**ID-keyed dict (`merged`)** in Exercise 3 makes deduplication O(1) per lookup instead of O(n) per lookup if searching a list. For a large dataset this matters.

### Use of Python commands and APIs
- `str.split()`, `str.strip()`, `str.join()` — only string functions as required
- `datetime.strptime()` — converts date strings to comparable objects for sorting (sorting ISO 8601 strings directly would also work since `YYYY-MM-DD` sorts lexicographically, but explicit parsing is clearer and safer)
- `sorted()` with `key=lambda` — clean, non-destructive sort
- `with open()` context manager — ensures files are closed even on error
- No `csv` module, no `pandas` — constraint respected throughout

### Comparison with activity-2.py starter code
The provided `activity-2.py` had three issues worth noting:
1. Column indices were hardcoded (`values[0]`, `values[1]`...) — breaks immediately when the update file has a different column order
2. The file was nested inside the `with` block, meaning the `import datetime` was also nested (valid in Python but non-standard)
3. No malformed row detection — a row with a missing comma would silently produce wrong data

This solution addresses all three by using dynamic header mapping, top-level imports, and explicit column count validation.